In [1]:
#pip install pyarrow
#pip install kagglehub
#pip install spark

In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("arvindnagaonkar/flight-delay")

print("Path to dataset files:", path)

/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: /home/codespace/.cache/kagglehub/datasets/arvindnagaonkar/flight-delay/versions/2


# We Can Explore with Pandas Dataframes, though this takes up more space in our virtual environment (Github Codespaces)

In [3]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import os

# List files in the dataset directory to find the CSV file
files = os.listdir(path)
print("Files in dataset directory:", files)

data_files = [f for f in files if f.endswith('.parquet')] 
if data_files: 
    data_file_path = os.path.join(path, data_files[0])

    schema = pq.read_schema(data_file_path)
    print("Available columns:")
    print(schema.names)

    cols_to_read = ['FlightDate', 'OriginCityName', 'DestCityName']  # columns grabbed

    table = pq.read_table(
        data_file_path,
        columns=cols_to_read
    )

    # Then slice just a few rows
    #   without this my github codespace instance crashes
    small_df = table.slice(0, 1000).to_pandas()
    print(small_df.head())
else:
    print("No data file found in the dataset directory.")

Files in dataset directory: ['features_added.parquet', 'Flight_Delay.parquet']
Available columns:
['Year', 'Month', 'DayofMonth', 'FlightDate', 'Marketing_Airline_Network', 'OriginCityName', 'DestCityName', 'DepTime', 'DepDelay', 'DepDelayMinutes', 'TaxiOut', 'TaxiIn', 'ArrTime', 'ArrDelay', 'ArrDelayMinutes', 'CRSElapsedTime', 'ActualElapsedTime', 'AirTime', 'Distance', 'DistanceGroup', 'CarrierDelay', 'WeatherDelay', 'NASDelay', 'SecurityDelay', 'LateAircraftDelay', 'DayofWeek', 'Holidays', 'CRSDepTimeMinute', 'CRSDepTimeHour', 'WheelsOffMinute', 'WheelsOffHour', 'CRSArrTimeMinute', 'CRSArrTimeHour', 'WheelsOnMinute', 'WheelsOnHour', 'CRSDepTimeHourDis', 'WheelsOffHourDis', 'CRSArrTimeHourDis', 'WheelsOnHourDis', 'CRSElapsedTimeGorup', '__index_level_0__']
  FlightDate OriginCityName    DestCityName
0 2018-01-15     Newark, NJ  Charleston, SC
1 2018-01-16     Newark, NJ  Charleston, SC
2 2018-01-17     Newark, NJ  Charleston, SC
3 2018-01-18     Newark, NJ  Charleston, SC
4 2018-01-2

# We will do the rest of the exploration with Spark

In [4]:
from pyspark.sql import SparkSession

# Start Spark session
spark = SparkSession.builder \
    .appName("FlightDelayAnalysis") \
    .getOrCreate()

# Get our parquet file path
files = os.listdir(path)
print("Files in dataset directory:", files)
data_files = [f for f in files if f.endswith('.parquet')] 
data_file_path = os.path.join(path, data_files[0])
df = spark.read.parquet(data_file_path)

# Show schema (column names and types)
df.printSchema()
df.show(10)

# Select specific columns and filter
df.select("FlightDate", "OriginCityName", "DestCityName").show(5)

print("Total rows:", df.count())

# Run basic aggregation
df.groupBy("OriginCityName").count().orderBy("count", ascending=False).show(10)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/10/07 13:46:26 WARN Utils: Your hostname, codespaces-90a6fa, resolves to a loopback address: 127.0.0.1; using 10.0.13.150 instead (on interface eth0)
25/10/07 13:46:26 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/10/07 13:46:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Files in dataset directory: ['features_added.parquet', 'Flight_Delay.parquet']


root
 |-- Year: long (nullable = true)
 |-- Month: long (nullable = true)
 |-- DayofMonth: long (nullable = true)
 |-- FlightDate: timestamp_ntz (nullable = true)
 |-- Marketing_Airline_Network: string (nullable = true)
 |-- OriginCityName: string (nullable = true)
 |-- DestCityName: string (nullable = true)
 |-- DepTime: double (nullable = true)
 |-- DepDelay: double (nullable = true)
 |-- DepDelayMinutes: double (nullable = true)
 |-- TaxiOut: double (nullable = true)
 |-- TaxiIn: double (nullable = true)
 |-- ArrTime: double (nullable = true)
 |-- ArrDelay: double (nullable = true)
 |-- ArrDelayMinutes: double (nullable = true)
 |-- CRSElapsedTime: double (nullable = true)
 |-- ActualElapsedTime: double (nullable = true)
 |-- AirTime: double (nullable = true)
 |-- Distance: double (nullable = true)
 |-- DistanceGroup: long (nullable = true)
 |-- CarrierDelay: double (nullable = true)
 |-- WeatherDelay: double (nullable = true)
 |-- NASDelay: double (nullable = true)
 |-- SecurityDel

25/10/07 13:46:35 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+----+-----+----------+-------------------+-------------------------+--------------+--------------+-------+--------+---------------+-------+------+-------+--------+---------------+--------------+-----------------+-------+--------+-------------+------------+------------+--------+-------------+-----------------+---------+--------+----------------+--------------+---------------+-------------+----------------+--------------+--------------+------------+-----------------+----------------+-----------------+---------------+-------------------+-----------------+
|Year|Month|DayofMonth|         FlightDate|Marketing_Airline_Network|OriginCityName|  DestCityName|DepTime|DepDelay|DepDelayMinutes|TaxiOut|TaxiIn|ArrTime|ArrDelay|ArrDelayMinutes|CRSElapsedTime|ActualElapsedTime|AirTime|Distance|DistanceGroup|CarrierDelay|WeatherDelay|NASDelay|SecurityDelay|LateAircraftDelay|DayofWeek|Holidays|CRSDepTimeMinute|CRSDepTimeHour|WheelsOffMinute|WheelsOffHour|CRSArrTimeMinute|CRSArrTimeHour|WheelsOnMinute|W

+-------------------+--------------+--------------+
|         FlightDate|OriginCityName|  DestCityName|
+-------------------+--------------+--------------+
|2018-01-15 00:00:00|    Newark, NJ|Charleston, SC|
|2018-01-16 00:00:00|    Newark, NJ|Charleston, SC|
|2018-01-17 00:00:00|    Newark, NJ|Charleston, SC|
|2018-01-18 00:00:00|    Newark, NJ|Charleston, SC|
|2018-01-20 00:00:00|    Newark, NJ|Charleston, SC|
+-------------------+--------------+--------------+
only showing top 5 rows
Total rows: 30132631


+--------------------+-------+
|      OriginCityName|  count|
+--------------------+-------+
|         Chicago, IL|1708956|
|         Atlanta, GA|1477844|
|Dallas/Fort Worth...|1166054|
|          Denver, CO|1131732|
|        New York, NY|1106836|
|       Charlotte, NC| 966926|
|         Houston, TX| 894020|
|     Los Angeles, CA| 877500|
|      Washington, DC| 875581|
|         Seattle, WA| 731920|
+--------------------+-------+
only showing top 10 rows
